# 07 · Generate expression on a developmental brain atlas

In this tutorial, you will use ten **E14 reference slices** to generate expression on three sections of the **E15.5 DevCCF atlas**.
You will then display those sections at their atlas z positions to view the result in 3D.

This example uses Study 07's 550-gene panel and assignment randomness of **0.4**.
Each section uses seed `2026 + its original z index`.
The full study generates all 158 E15.5 sections and, in a separate run, 202 E18.5 sections.

Follow [case 07 setup](README.md#case-07-public-sources-and-prepared-inputs) for the prepared reference labels and atlas files.
Generation requires Torch/CUDA. Loading and fitting all ten reference slices also requires substantial RAM, even when generating only three sections.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
import FEAST

TUTORIAL = Path.cwd() if Path.cwd().name == "tutorial" else Path.cwd() / "tutorial"
sys.path.insert(0, str(TUTORIAL))
from _utils import data_root, load_counts, gene_summary, gene_values, spatial_panel
DATA = data_root()

OUT = TUTORIAL / "outputs" / "07"
OUT.mkdir(parents=True, exist_ok=True)
print("FEAST", FEAST.__version__)

### 1. Load the reference slices and atlas sections

The reference slices contain measured expression and broad anatomical labels in `obs['region']`.
The atlas sections use the same region names but contain only coordinates and voxel IDs.
This shared vocabulary lets FEAST match reference regions to atlas regions.

Here we use the E14 references and E15.5 atlas.
The E18.5 example requires five E18M references and its own randomness calibration; see the full Study 07 workflow for that setup.

In [ ]:
import gzip
import json
from FEAST.de_novo import fit_reference

AGE = "E15.5"
Z_INDICES = [47, 78, 109]  # Original sorted atlas indices; fixed before generation.
reference_paths = sorted((DATA / "07/references").glob("*E14*.h5ad"))
assert len(reference_paths) == 10, "Supply all ten prepared E14 references (see README.md)."
references = [load_counts(path, "region") for path in reference_paths]
for path, reference in zip(reference_paths, references):
    reference.uns["reference_name"] = path.stem
with gzip.open(DATA / "07/E15.5.blueprint.json.gz", "rt") as handle:
    payload = json.load(handle)
assert payload["metadata"]["expression_source"] == "none"
entries = [payload["slices"][i] for i in Z_INDICES]
pd.DataFrame([{k: entry[k] for k in ["z_index", "z_world", "n_spots"]} for entry in entries])

### 2. Fit the references once and generate each section

Fit one reference model using all ten slices, then reuse it for the three selected atlas sections.
For each section, FEAST performs a 2D conditional transfer, as in notebook 05.
We then attach its atlas z coordinate to place the result in 3D.

Sections are generated separately, with no smoothing between them.
The code below keeps Study 07's log-domain transport solver, float64 precision and convergence requirements.

In [ ]:
model = fit_reference(references, label_key="region", config=FEAST.ReferenceFitConfig(
    min_gene_spots=1, min_gene_mean=0.0, max_gene_zero_prop=1.0,
))
assert len(model.gene_names) == 550
transport = FEAST.SimulationConfig(
    assignment_randomness=0.4, sinkhorn_method="sinkhorn_log",
    transport_backend="torch", transport_device="cuda:0", transport_dtype="float64",
    epsilon=0.05, sinkhorn_iter=1000, sinkhorn_tol=1e-5,
    unbalanced_transport=True, reg_m=5.0, transport_nonconvergence="raise",
    max_transport_pairs=25_000_000, quantile_field_mode="latent_reference",
    store_latent_scores=False, store_quantiles=False,
)

In [ ]:
generated_sections = []
for entry in entries:
    z_index, z = entry["z_index"], entry["z_world"]
    ids = [f"devccf-e15_5-z{z_index:03d}-j{entry['voxel_j']:03d}-i{i:03d}-k{k:03d}"
           for i, k in zip(entry["voxel_i"], entry["voxel_k"])]
    obs = pd.DataFrame({"spot_id": ids, "region": entry["region"], "z": z})
    blueprint = FEAST.SliceBlueprint(
        coordinates=np.column_stack([entry["x"], entry["y"]]),
        domain_map=np.asarray(entry["region"]), obs=obs,
        grid_type="devccf_coronal", technology="MERFISH",
        metadata={"age": AGE, "z_world": z, "expression_source": "none"},
    )
    generated = FEAST.simulate(model, target=blueprint, marginal_model="empirical_reference",
                               transport=transport, seed=2026 + z_index)
    assert generated.obs["spot_id"].tolist() == ids
    generated.obs_names = generated.obs["spot_id"].astype(str)
    generated.obsm["spatial_3d"] = np.column_stack([generated.obsm["spatial"], np.full(generated.n_obs, z)])
    generated.write_h5ad(OUT / f"E15.5_z{z_index:03d}.h5ad")
    generated_sections.append(generated)

### 3. View anatomy and expression in 3D

The left panel colors atlas locations by region. The right panel colors those same locations by generated **Reln** expression.
Both panels show every voxel in the three selected sections.

The gaps between planes are expected: we selected three z levels for this tutorial rather than generating the full volume.

In [ ]:
xyz = np.vstack([section.obsm["spatial_3d"] for section in generated_sections])
regions = np.concatenate([section.obs["region"].astype(str).to_numpy() for section in generated_sections])
values = np.concatenate([np.log1p(gene_values(section, "Reln")) for section in generated_sections])
fig = plt.figure(figsize=(12, 5), layout="constrained")
ax = fig.add_subplot(121, projection="3d")
labels = sorted(set(regions))
colors = plt.get_cmap("tab10")
for i, label in enumerate(labels):
    ax.scatter(*xyz[regions == label].T, s=1, color=colors(i), label=label, rasterized=True)
ax.legend(fontsize=6, markerscale=3, loc="upper left")
ax.set(title="Atlas regions · three selected sections", xlabel="x", ylabel="y", zlabel="z")
ax = fig.add_subplot(122, projection="3d")
artist = ax.scatter(*xyz.T, c=values, s=1, cmap="viridis", vmin=0, rasterized=True)
fig.colorbar(artist, ax=ax, shrink=0.6, label="Generated Reln · log1p counts")
ax.set(title="Conditional expression", xlabel="x", ylabel="y", zlabel="z")
plt.show()

**How to read the plots:** DevCCF supplies the anatomy, while FEAST generates expression using the reference data and region labels.
There is no measured expression at these atlas locations, so the figure shows the generated pattern without establishing its accuracy.

The full reproduction workflow also checks coverage along the atlas axis and how expression changes between adjacent sections.

### Saved example

![DevCCF E15.5 levels 47, 78 and 109: atlas region labels and generated Reln expression, without cross-z smoothing.](assets/07.png)

DevCCF E15.5 levels 47, 78 and 109: atlas region labels and generated Reln expression, without cross-z smoothing.

This image comes from an earlier reproduction run. It is not a new result from this notebook. A fresh run may differ with the software version and environment.

<details>
<summary>Image sources</summary>

Rendered on 2026-09-04 from these existing files in `FEAST_reproduce`, using the plotting code above:

- `07_3d_transfer/outputs/final/E15.5/z047__-3.180.h5ad`
- `07_3d_transfer/outputs/final/E15.5/z078__-2.560.h5ad`
- `07_3d_transfer/outputs/final/E15.5/z109__-1.940.h5ad`

The source files were read without rerunning the simulations. These images do not show a new execution of the notebook. The original Study 07 run used its recorded provisional FEAST build.

</details>